# export_

> Export routes for single document and full database JSON file downloads

In [ ]:
#| default_exp routes.export_

In [ ]:
#| export
import json
import re
from typing import Dict, Callable, Tuple
from dataclasses import asdict
from datetime import datetime, timezone

from fasthtml.common import Response
from cjm_fasthtml_app_core.core.routing import APIRouter

from cjm_transcript_workflow_management.services.management import ManagementService
from cjm_transcript_workflow_management.routes.core import DEBUG_MANAGEMENT_ROUTES

## Helpers

In [ ]:
#| export
def _sanitize_filename(
    name:str,  # Raw filename string
) -> str:  # Filesystem-safe filename
    """Remove characters unsafe for filenames."""
    return re.sub(r'[^\w\s\-.]', '', name).strip().replace(' ', '_')

In [ ]:
assert _sanitize_filename('1. Laying Plans') == '1._Laying_Plans'
assert _sanitize_filename('test/file:name') == 'testfilename'
assert _sanitize_filename('  hello  ') == 'hello'
print("Sanitize OK")

Sanitize OK


In [ ]:
#| export
def _bundle_to_json_response(
    bundle_dict:dict,  # Serialized ExportBundle
    filename:str,  # Download filename (e.g., "document.json")
) -> Response:  # Starlette Response with JSON content and download headers
    """Create a file download response from an export bundle dict."""
    json_bytes = json.dumps(bundle_dict, indent=2, ensure_ascii=False).encode('utf-8')
    return Response(
        content=json_bytes,
        media_type='application/json',
        headers={'Content-Disposition': f'attachment; filename="{filename}"'},
    )

## Router Initialization

In [ ]:
#| export
def init_export_router(
    service:ManagementService,  # Service for graph queries
    prefix:str,  # Route prefix (e.g., "/manage/export")
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, routes dict)
    """Initialize export routes for single document and full database downloads."""
    router = APIRouter(prefix=prefix)
    routes = {}
    
    @router
    async def export_document(request, doc_id:str=""):
        """Export a single document as a JSON file download."""
        if DEBUG_MANAGEMENT_ROUTES:
            print(f"[ROUTES] export_document called for {doc_id}")
        
        if not doc_id:
            return Response(content=b'{"error": "No document ID provided"}',
                           media_type='application/json', status_code=400)
        
        bundle = await service.export_document_async(doc_id)
        if bundle is None:
            return Response(content=b'{"error": "Document not found"}',
                           media_type='application/json', status_code=404)
        
        # Build filename from document title
        doc_title = "document"
        for node in bundle.graph.get("nodes", []):
            if node.get("label") == "Document":
                doc_title = node.get("properties", {}).get("title", "document")
                break
        
        filename = f"{_sanitize_filename(doc_title)}.json"
        
        if DEBUG_MANAGEMENT_ROUTES:
            print(f"[ROUTES] Exporting as {filename}")
        
        return _bundle_to_json_response(asdict(bundle), filename)
    
    routes["export_document"] = export_document
    
    @router
    async def export_all(request):
        """Export the entire graph database as a JSON file download."""
        if DEBUG_MANAGEMENT_ROUTES:
            print("[ROUTES] export_all called")
        
        bundle = await service.export_all_async()
        if bundle is None:
            return Response(content=b'{"error": "Export failed"}',
                           media_type='application/json', status_code=500)
        
        date_str = datetime.now(timezone.utc).strftime('%Y%m%d')
        filename = f"graph_export_{date_str}.json"
        
        if DEBUG_MANAGEMENT_ROUTES:
            print(f"[ROUTES] Exporting all as {filename}, {bundle.document_count} docs")
        
        return _bundle_to_json_response(asdict(bundle), filename)
    
    routes["export_all"] = export_all
    
    return router, routes

In [ ]:
assert init_export_router is not None
print("routes.export_ imports OK")

routes.export_ imports OK


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()